# 🏥 TrialMatch AI — Criteria Parser Fine-Tuning

**Objective**: Fine-tune BioBERT using **QLoRA (PEFT)** to decompose eligibility criteria into structured rules.

**Method**: QLoRA (4-bit quantization + Low-Rank Adaptation) — trains only ~0.5% of parameters.

**Why QLoRA over full fine-tuning?**
- Fits on free Colab T4 (16GB VRAM) — full FT would OOM on larger models
- 10x less memory, 3x faster training
- Produces a tiny adapter (~10MB) instead of a full model copy (~400MB)
- Near-identical accuracy for classification/structured extraction tasks

**Runtime**: GPU required → Runtime → Change runtime type → **T4 GPU**

---
### Training Pipeline
```
train.jsonl → Tokenize → BioBERT + QLoRA Adapter → Train → Evaluate → Merge → Export
```

In [ ]:
# ═══ Step 1: Install Dependencies ═══
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes scikit-learn trl

In [ ]:
# ═══ Step 2: Verify GPU ═══
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ═══ Step 3: Upload Training Data ═══
# Upload: fine_tuning/data/criteria_parser/train.jsonl
from google.colab import files
print("Upload train.jsonl (from fine_tuning/data/criteria_parser/)")
uploaded = files.upload()

In [ ]:
# ═══ Step 4: Configuration ═══
CONFIG = {
    # Model
    "base_model": "dmis-lab/biobert-v1.1",  # Options: "dmis-lab/biobert-v1.1", "emilyalsentzer/Bio_ClinicalBERT"
    "max_seq_length": 256,
    
    # QLoRA / PEFT
    "lora_r": 16,             # LoRA rank (higher = more params, more capacity)
    "lora_alpha": 32,         # Scaling factor (typically 2x rank)
    "lora_dropout": 0.05,     # Dropout on LoRA layers
    "target_modules": ["query", "value"],  # Which attention layers to adapt
    "quantization_bits": 4,   # 4-bit quantization (QLoRA)
    
    # Training
    "num_epochs": 15,
    "batch_size": 16,
    "learning_rate": 2e-4,    # Higher LR for LoRA (only training adapter)
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    
    # Labels: 6 classes (3 complexity × 2 criteria_type)
    # simple_inclusion=0, simple_exclusion=1, compound_inclusion=2,
    # compound_exclusion=3, conditional_inclusion=4, conditional_exclusion=5
    "num_labels": 6,
    "label_names": [
        "simple_inclusion", "simple_exclusion",
        "compound_inclusion", "compound_exclusion",
        "conditional_inclusion", "conditional_exclusion"
    ],
    
    # Output
    "output_dir": "./criteria_parser_qlora",
    "merged_dir": "./criteria_parser_merged",
}

print("✅ Config ready")
print(f"   Model: {CONFIG['base_model']}")
print(f"   LoRA rank: {CONFIG['lora_r']}, alpha: {CONFIG['lora_alpha']}")
print(f"   Quantization: {CONFIG['quantization_bits']}-bit (QLoRA)")
print(f"   Epochs: {CONFIG['num_epochs']}, LR: {CONFIG['learning_rate']}")

In [ ]:
# ═══ Step 5: Load and Preprocess Data ═══
import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load JSONL
examples = []
with open("train.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        # Map (complexity, criteria_type) → label index
        complexity_map = {"simple": 0, "compound": 1, "conditional": 2}
        type_map = {"inclusion": 0, "exclusion": 1}
        
        output = item["output"]
        complexity = output.get("complexity", "simple")
        criteria_type = output.get("criteria_type", "inclusion")
        label = complexity_map.get(complexity, 0) * 2 + type_map.get(criteria_type, 0)
        
        examples.append({
            "text": item["input"],
            "label": label,
            "full_output": json.dumps(output),  # Keep for reference
        })

print(f"Total examples: {len(examples)}")

# Train/val split (80/20)
if len(examples) > 10:
    train_data, val_data = train_test_split(examples, test_size=0.2, random_state=42, stratify=[e["label"] for e in examples] if len(set(e["label"] for e in examples)) > 1 else None)
else:
    # Too few examples for stratified split
    train_data, val_data = train_test_split(examples, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"\nLabel distribution:")
for i, name in enumerate(CONFIG["label_names"]):
    count = sum(1 for e in examples if e["label"] == i)
    print(f"  {i}: {name} → {count} examples")

print(f"\nSample: {train_dataset[0]['text'][:80]}... → label {train_dataset[0]['label']}")

In [ ]:
# ═══ Step 6: Tokenize ═══
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    )

train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "full_output"])
val_tokenized = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "full_output"])

# Set format for PyTorch
train_tokenized.set_format("torch")
val_tokenized.set_format("torch")

print(f"✅ Tokenized — max_length: {CONFIG['max_seq_length']}")
print(f"   Vocab size: {tokenizer.vocab_size}")

In [ ]:
# ═══ Step 7: Load Base Model with 4-bit Quantization ═══
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig

# QLoRA: Load model in 4-bit precision
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 (best for fine-tuning)
    bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16
    bnb_4bit_use_double_quant=True,        # Double quantization (saves more memory)
)

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=CONFIG["num_labels"],
    quantization_config=bnb_config,
    device_map="auto",
)

# Print model size
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {CONFIG['base_model']}")
print(f"   Total parameters: {total_params:,}")
print(f"   Quantization: 4-bit NF4 (QLoRA)")
print(f"   Memory: ~{total_params * 0.5 / 1e9:.2f} GB (4-bit)")

In [ ]:
# ═══ Step 8: Apply LoRA Adapter (PEFT) ═══
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,           # Sequence classification
    r=CONFIG["lora_r"],                    # Rank
    lora_alpha=CONFIG["lora_alpha"],        # Scaling
    lora_dropout=CONFIG["lora_dropout"],    # Dropout
    target_modules=CONFIG["target_modules"],# Which layers to adapt
    bias="none",                            # Don't train bias terms
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Print trainable vs total params
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct = 100 * trainable_params / total_params

print(f"✅ LoRA adapter applied")
print(f"   Trainable parameters: {trainable_params:,} ({pct:.2f}%)")
print(f"   Frozen parameters:    {total_params - trainable_params:,} ({100-pct:.2f}%)")
print(f"   LoRA rank: {CONFIG['lora_r']}, alpha: {CONFIG['lora_alpha']}")
print(f"   Target modules: {CONFIG['target_modules']}")
print(f"\n   💡 Only training {pct:.2f}% of parameters — this is the power of QLoRA")

In [ ]:
# ═══ Step 9: Training Setup ═══
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1_score["f1"]}

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=5,
    fp16=True,                     # Mixed precision (works with QLoRA)
    optim="paged_adamw_8bit",      # Memory-efficient optimizer for QLoRA
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("✅ Trainer ready")
print(f"   Optimizer: paged_adamw_8bit (QLoRA-optimized)")
print(f"   Precision: fp16 mixed")
print(f"   Epochs: {CONFIG['num_epochs']}, Batch: {CONFIG['batch_size']}")

In [ ]:
# ═══ Step 10: Train! ═══
print("🚀 Starting QLoRA fine-tuning...\n")
train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Final training loss: {train_result.training_loss:.4f}")
print(f"   Total steps: {train_result.global_step}")

In [ ]:
# ═══ Step 11: Evaluate ═══
eval_results = trainer.evaluate()

print("\n" + "═" * 50)
print("📊 EVALUATION RESULTS")
print("═" * 50)
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")
print("═" * 50)

In [ ]:
# ═══ Step 12: Merge LoRA Adapter into Base Model ═══
# This creates a standalone model that doesn't need PEFT at inference time
from peft import PeftModel

print("Merging LoRA adapter into base model...")

# Reload base model in full precision for merging
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=CONFIG["num_labels"],
    torch_dtype=torch.float16,
)

# Load the trained LoRA adapter
# Find the best checkpoint
import os
checkpoints = [d for d in os.listdir(CONFIG["output_dir"]) if d.startswith("checkpoint-")]
if checkpoints:
    best_checkpoint = os.path.join(CONFIG["output_dir"], sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1])
else:
    best_checkpoint = CONFIG["output_dir"]

print(f"Loading adapter from: {best_checkpoint}")
model_with_lora = PeftModel.from_pretrained(base_model, best_checkpoint)

# Merge LoRA weights into base model
merged_model = model_with_lora.merge_and_unload()

# Save merged model
merged_model.save_pretrained(CONFIG["merged_dir"])
tokenizer.save_pretrained(CONFIG["merged_dir"])

print(f"✅ Merged model saved to: {CONFIG['merged_dir']}")
print(f"   This model runs WITHOUT peft/bitsandbytes at inference")

In [ ]:
# ═══ Step 13: Save Training Metrics ═══
import json

metrics = {
    "model": CONFIG["base_model"],
    "method": "QLoRA (PEFT)",
    "quantization": f"{CONFIG['quantization_bits']}-bit NF4",
    "lora_rank": CONFIG["lora_r"],
    "lora_alpha": CONFIG["lora_alpha"],
    "target_modules": CONFIG["target_modules"],
    "trainable_params_pct": round(pct, 2),
    "epochs": CONFIG["num_epochs"],
    "learning_rate": CONFIG["learning_rate"],
    "optimizer": "paged_adamw_8bit",
    "train_loss": round(train_result.training_loss, 4),
    "eval_results": {k: round(v, 4) if isinstance(v, float) else v for k, v in eval_results.items()},
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "label_names": CONFIG["label_names"],
}

with open(f"{CONFIG['merged_dir']}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Also save the LoRA config for documentation
with open(f"{CONFIG['merged_dir']}/lora_config.json", "w") as f:
    json.dump({
        "r": CONFIG["lora_r"],
        "alpha": CONFIG["lora_alpha"],
        "dropout": CONFIG["lora_dropout"],
        "target_modules": CONFIG["target_modules"],
        "quantization": f"{CONFIG['quantization_bits']}-bit NF4",
        "method": "QLoRA via PEFT",
    }, f, indent=2)

print("✅ Metrics saved")
print(json.dumps(metrics, indent=2))

In [ ]:
# ═══ Step 14: Test Inference ═══
from transformers import pipeline as hf_pipeline

# Load the merged model for inference
classifier = hf_pipeline(
    "text-classification",
    model=CONFIG["merged_dir"],
    tokenizer=CONFIG["merged_dir"],
    device=0 if torch.cuda.is_available() else -1,
)

# Test examples
test_criteria = [
    "Patients must be at least 18 years of age.",
    "No prior treatment with any anti-PD-1 antibody.",
    "ANC >= 1500/uL, platelets >= 100000/uL, hemoglobin >= 9.0 g/dL.",
    "Total bilirubin <= 1.5 x ULN OR direct bilirubin <= ULN if total > 1.5 x ULN.",
]

print("\n🧪 INFERENCE TEST")
print("─" * 60)
for text in test_criteria:
    result = classifier(text)
    label_idx = int(result[0]["label"].split("_")[-1])
    label_name = CONFIG["label_names"][label_idx] if label_idx < len(CONFIG["label_names"]) else result[0]["label"]
    print(f"\nInput: {text[:70]}...")
    print(f"  → {label_name} (confidence: {result[0]['score']:.3f})")

In [ ]:
# ═══ Step 15: Zip and Download ═══
import shutil

# Zip the merged model
shutil.make_archive("criteria_parser_v1", "zip", ".", CONFIG["merged_dir"])

print("\n" + "═" * 60)
print("📦 MODEL READY FOR DOWNLOAD")
print("═" * 60)
print(f"\nFiles in {CONFIG['merged_dir']}/")
for f in sorted(os.listdir(CONFIG["merged_dir"])):
    size = os.path.getsize(os.path.join(CONFIG["merged_dir"], f)) / 1024
    print(f"   {f} ({size:.1f} KB)")

print(f"\n📥 Downloading criteria_parser_v1.zip...")
print(f"\n🎯 NEXT STEPS:")
print(f"   1. Download the zip file")
print(f"   2. Unzip into: trialmatch-ai/fine_tuning/models/criteria_parser/")
print(f"   3. The app will auto-detect it in the sidebar model picker")

from google.colab import files
files.download("criteria_parser_v1.zip")